# PolERo: train on Google Colab

**Before running:** menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU** → Save.
(With Colab Pro, *L4* or *A100* are faster; the batch size and precision adapt automatically: fp16 on T4, bf16 on L4/A100.)

1. Put `polero-src.zip` in the root of your Google Drive (`My Drive/polero-src.zip`).
2. **Runtime → Run all** and allow access to Google Drive.
3. Results are saved to `My Drive/polero_artifacts` after every fold. If Colab disconnects, just **Run all** again: finished folds are skipped.
4. At the end: `My Drive/polero_out/submission.csv` and one zip per experiment.

Free Colab has a single T4 and limited session time, so the three experiments may need two or three sessions; the order in `CONFIGS` is the priority order.

In [ ]:
# ---- settings ----
SMOKE = False   # True = 2-minute setup check (2 folds x 2 steps, tiny model). Then set back to False.
CONFIGS = [
    "configs/robert-large_base.json",   # main model (RoBERT-large, research prior)
    "configs/robert-large_ctx.json",    # + previous question and president/context metadata
    "configs/xlmr-large_base.json",     # XLM-R-large, for ensemble diversity
]
FOLDS = "0,1,2,3,4,5,6"


In [ ]:
import glob, os, shutil, subprocess, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive'
ZIP = D + '/polero-src.zip'
if not os.path.exists(ZIP):
    from google.colab import files
    print('polero-src.zip not found in My Drive; upload it now')
    ZIP = '/content/' + list(files.upload())[0]
WORK = '/content/polero'
shutil.rmtree(WORK, ignore_errors=True)
zipfile.ZipFile(ZIP).extractall(WORK)
os.chdir(WORK)
OUT = D + '/polero_out'
ARTIFACTS = D + '/polero_artifacts'   # on Drive: survives disconnects, enables resume
os.makedirs(ARTIFACTS, exist_ok=True)
shutil.copy('artifacts/folds.csv', ARTIFACTS + '/folds.csv')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.23.0', 'sentencepiece==0.2.3'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'], check=True)
import torch, transformers
assert torch.cuda.is_available(), 'No GPU: enable a GPU accelerator first (see the first cell).'
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}, {p.total_memory / 1e9:.0f} GB')
print('torch', torch.__version__, '| transformers', transformers.__version__)


In [ ]:
cmd = [sys.executable, '-u', 'scripts/kaggle_run.py', '--folds', FOLDS, '--out-dir', OUT, '--artifacts', ARTIFACTS]
for c in CONFIGS:
    cmd += ['--config', c]
if SMOKE:
    cmd.append('--smoke')
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='')
print('exit code', p.wait())


In [ ]:
# per-epoch progress of every fold, and the files to download
for f in sorted(glob.glob(ARTIFACTS + '/*/train.log')):
    print('=' * 20, f)
    print(open(f, encoding='utf-8').read()[-2500:])
print(sorted(glob.glob(OUT + '/*')))
